<a href="https://colab.research.google.com/github/draginverse/dragin-healthcare/blob/feature%2Fg-retriever/scripts/retrieval/similarity_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

In [7]:
# Toy graph data
toy_graphs = [
    [
        ("asthma", "caused_by", "allergens"),
        ("inhaler", "treats", "asthma"),
        ("asthma", "symptom", "shortness of breath")
    ],
    [
        ("copd", "risk_factor", "smoking"),
        ("oxygen therapy", "treats", "copd"),
        ("copd", "symptom", "chronic cough")
    ],
    [
        ("bronchitis", "caused_by", "virus"),
        ("bronchitis", "symptom", "chest discomfort"),
        ("rest", "helps_with", "bronchitis")
    ],
    [
        ("pneumonia", "caused_by", "bacteria"),
        ("antibiotics", "treats", "pneumonia"),
        ("pneumonia", "symptom", "fever")
    ],
    [
        ("covid-19", "affects", "lungs"),
        ("vaccine", "prevents", "covid-19"),
        ("covid-19", "symptom", "loss of smell")
    ]
]


In [11]:
# --- Graph Text Representation ---
def graph_to_text(graph):
    return " ".join([f"{s} {r} {o}." for s, r, o in graph])

In [12]:
# --- Graph Embedder ---
class GraphEmbedder:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)

    def embed_graphs(self, graphs, text_fn=graph_to_text):
        graph_texts = [text_fn(g) for g in graphs]
        embeddings = self.model.encode(graph_texts, convert_to_tensor=True)
        return graph_texts, embeddings

    def embed_query(self, query):
        return self.model.encode(query, convert_to_tensor=True)


In [13]:
# --- Similarity-Based Retriever ---
class GraphRetriever:
    def __init__(self, graphs, graph_texts, graph_embeddings):
        self.graphs = graphs
        self.graph_texts = graph_texts
        self.graph_embeddings = graph_embeddings

    def retrieve(self, query_embedding, top_k=1):
        scores = util.cos_sim(query_embedding, self.graph_embeddings)[0]
        top_k = min(top_k, len(self.graphs))
        top_results = scores.topk(k=top_k)
        top_indices = top_results.indices.tolist()
        top_scores = top_results.values.tolist()
        return [(self.graphs[i], self.graph_texts[i], top_scores[idx]) for idx, i in enumerate(top_indices)]

In [14]:
if __name__ == "__main__":
    embedder = GraphEmbedder()
    graph_texts, graph_embeddings = embedder.embed_graphs(toy_graphs)

    retriever = GraphRetriever(toy_graphs, graph_texts, graph_embeddings)

    query = "i need a cure for asthma"
    query_embedding = embedder.embed_query(query)

    top_k_results = retriever.retrieve(query_embedding, top_k=3)

    print(f"\nQuery: {query}\n")

    for rank, (graph, text, score) in enumerate(top_k_results, 1):
        print(f"\nTop-{rank} Graph (Score: {score:.4f}):\n{text}")
        for triple in graph:
            print(triple)


Query: i need a cure for asthma


Top-1 Graph (Score: 0.5395):
asthma caused_by allergens. inhaler treats asthma. asthma symptom shortness of breath.
('asthma', 'caused_by', 'allergens')
('inhaler', 'treats', 'asthma')
('asthma', 'symptom', 'shortness of breath')

Top-2 Graph (Score: 0.3641):
copd risk_factor smoking. oxygen therapy treats copd. copd symptom chronic cough.
('copd', 'risk_factor', 'smoking')
('oxygen therapy', 'treats', 'copd')
('copd', 'symptom', 'chronic cough')

Top-3 Graph (Score: 0.3439):
covid-19 affects lungs. vaccine prevents covid-19. covid-19 symptom loss of smell.
('covid-19', 'affects', 'lungs')
('vaccine', 'prevents', 'covid-19')
('covid-19', 'symptom', 'loss of smell')
